# 0. Preparation

In [1]:
import os
import json
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt

In [5]:
data_path = r"C:\Users\User\OneDrive\Desktop\RIAW\irwa_search_engine_G_011\project_progress\processed_dataset.csv"

df = pd.read_csv(
    data_path,
    engine="python",
    on_bad_lines="skip"
)
df.head()

,pid,url,processed_text,title,description,brand_facet,category_facet,subcategory_facet,seller_facet,discount,selling_price,actual_price,average_rating,attributes
0,TKPFCZ9EA7H5FYZH,https://www.flipkart.com/yorker-solid-men-mult...,"['solid', 'women', 'multicolor', 'track', 'pan...",Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,69.0,921.0,2999.0,3.9,"['elast', 'side', 'pocket', 'cotton', 'blend',..."
1,TKPFCZ9EJZV2UVRZ,https://www.flipkart.com/yorker-solid-men-blue...,"['solid', 'men', 'blue', 'track', 'pant', 'yor...",Solid Men Blue Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,66.0,499.0,1499.0,3.9,"['drawstr', 'elast', 'side', 'pocket', 'cotton..."
2,TKPFCZ9EHFCY5Z4Y,https://www.flipkart.com/yorker-solid-men-mult...,"['solid', 'men', 'multicolor', 'track', 'pant'...",Solid Men Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,68.0,931.0,2999.0,3.9,"['elast', 'side', 'pocket', 'cotton', 'blend',..."
3,TKPFCZ9ESZZ7YWEF,https://www.flipkart.com/yorker-solid-men-mult...,"['solid', 'women', 'multicolor', 'track', 'pan...",Solid Women Multicolor Track Pants,Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,69.0,911.0,2999.0,3.9,"['elast', 'side', 'pocket', 'cotton', 'blend',..."
4,TKPFCZ9EVXKBSUD7,https://www.flipkart.com/yorker-solid-men-brow...,"['solid', 'women', 'brown', 'gray', 'track', '...","Solid Women Brown, Grey Track Pants",Yorker trackpants made from 100% rich combed c...,york,clothing_and_accessories,bottomwear,shyam_enterprises,68.0,943.0,2999.0,3.9,"['drawstr', 'elast', 'side', 'pocket', 'cotton..."


In [6]:
import ast

# Function to convert string representation of list to actual list
def safe_literal_eval(val):
    if pd.isna(val):
        return []  # Empty list for NaN values
    try:
        # ast.literal_eval to convert "['a', 'b']" to ['a', 'b']
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return [] # Return empty list if conversion fails

print("Converting columns 'processed_text' and 'attributes' to lists...")
df['processed_text'] = df['processed_text'].apply(safe_literal_eval)
df['attributes'] = df['attributes'].apply(safe_literal_eval)

# Check the conversion
print(f"The type of 'processed_text' is now: {type(df.iloc[0]['processed_text'])}")
print(f"Example of 'processed_text': {df.iloc[0]['processed_text'][:5]}...")

Converting columns 'processed_text' and 'attributes' to lists...
The type of 'processed_text' is now: <class 'list'>
Example of 'processed_text': ['solid', 'women', 'multicolor', 'track', 'pant']...


In [7]:
# we need all the tokens in a single column for the inverted index, hence we concatenate the processed_text and attributes columns
df['tokens'] = df['processed_text'] + df['attributes']

# 1. Ranking

## 1.1 TF-IDF + cosine similarity

Since we already implemented this ranking method in part 2, we will reuse the code in order to later compare with BM25 and our own ranking method. 

In [8]:
with open(os.path.join("..", "..", "project_progress", "part_2", "inverted_index.json"), "r") as f:
    inverted_index = json.load(f)

with open(os.path.join("..", "..", "project_progress", "part_2", "idf_scores.json"), "r") as f:
    idf_scores = json.load(f)

In [9]:
def setup_preprocessing_tools():
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))
    stop_words.update(['made', 'wear', 'comfort', 'quality', 
                       'look', 'perfect', 'style', 'great', 'cool'])
    stop_words.discard('no')
    stop_words.discard('not')
    return stemmer, stop_words

In [10]:
import re
import string

def preprocess_query(text, stemmer, stop_words):
    """
    Preprocess natural text:
    - lowercase
    - remove punctuation/numbers
    - tokenize
    - remove stopwords and non-alphabetic tokens
    - stem 
    """
    if not isinstance(text, str):
        return []
    
    text = text.replace('-', ' ')

    # lowercase
    text = text.lower()

    # remove punctuation and digits
    text = re.sub(f"[{re.escape(string.punctuation)}0-9]", " ", text)

    # tokenize
    tokens = word_tokenize(text)

    # filter tokens (stopwords, non-alpha, short tokens)
    tokens = [w for w in tokens if w.isalpha() and w not in stop_words]

    # stem
    tokens = [stemmer.stem(w) for w in tokens]

    # normalize color terms 
    color_map = {'navy': 'blue', 'grey': 'gray', 'fucsia': 'pink', 'burgundy': 'red', 'violet': 'purple', 'beige': 'brown', 'magenta': 'pink', 'indigo': 'blue', 
                 'charcoal': 'gray', 'crimson': 'red', 'teal': 'green', 'lavender': 'purple', 'mustard': 'yellow', 'turquoise': 'blue', 'peach': 'orange'}

    tokens = [color_map.get(w, w) for w in tokens]

    return tokens

In [11]:
def calculate_log_tf(tokens):
    """Calculates log-normalized TF for each term in a document's token list."""
    counts = Counter(tokens)
    return {term: 1 + np.log(count) for term, count in counts.items()} #Don't need to handle log(0) since count is always >=1 because it's only done for terms that appear in tokens

# Apply this function to every document's tokens
df['tf_scores'] = df['tokens'].apply(calculate_log_tf)

# Check the TF scores for the first document
print("TF scores for first document:")
print(df.iloc[0]['tf_scores'])

TF scores for first document:
{'solid': np.float64(1.6931471805599454), 'women': np.float64(1.0), 'multicolor': np.float64(1.6931471805599454), 'track': np.float64(1.0), 'pant': np.float64(1.0), 'yorker': np.float64(1.0), 'trackpant': np.float64(1.0), 'rich': np.float64(1.6931471805599454), 'comb': np.float64(1.0), 'cotton': np.float64(1.6931471805599454), 'give': np.float64(1.0), 'design': np.float64(1.0), 'skin': np.float64(1.0), 'friendli': np.float64(1.0), 'fabric': np.float64(1.0), 'itch': np.float64(1.0), 'free': np.float64(1.0), 'waistband': np.float64(1.0), 'year': np.float64(1.0), 'round': np.float64(1.0), 'use': np.float64(1.0), 'proudli': np.float64(1.0), 'india': np.float64(1.0), 'elast': np.float64(1.0), 'side': np.float64(1.0), 'pocket': np.float64(1.0), 'blend': np.float64(1.0)}


In [12]:
def calculate_tfidf_L2_norm(tf_scores, idf_scores_global):
    """
    Calculates the TF-IDF vector (as a dict) and the L2-norm (length)
    of that vector for a single document.
    """
    tfidf_vector = {}
    sum_of_squares = 0.0
    
    for term, tf in tf_scores.items():
        # Only include terms that are in our global IDF dictionary
        if term in idf_scores_global:
            tfidf = tf * idf_scores_global[term] # we multiply each TF value by the global IDF score
            tfidf_vector[term] = tfidf
            sum_of_squares += tfidf**2 
            
    doc_length = np.sqrt(sum_of_squares) # we compute the L2 norm 
    return tfidf_vector, doc_length

# Apply the function to the 'tf_scores' column
# This returns a tuple (tfidf_vector, doc_length), so we split it into two new columns
tfidf_results = df['tf_scores'].apply(lambda tf: calculate_tfidf_L2_norm(tf, idf_scores))
df['tfidf_vector'] = tfidf_results.apply(lambda x: x[0]) # tfidf vector dictionary
df['doc_length'] = tfidf_results.apply(lambda x: x[1]) # document length

# Check the results for the first document
print("TF-IDF vector for first document:")
print(df.iloc[0]['tfidf_vector'])
print("\nDocument length for first document:")
print(df.iloc[0]['doc_length'])

TF-IDF vector for first document:
{'solid': np.float64(1.439853988827382), 'women': np.float64(0.7098818692235951), 'multicolor': np.float64(2.979595529588283), 'track': np.float64(2.738972111440796), 'pant': np.float64(2.6079921803942043), 'yorker': np.float64(7.107318642210598), 'trackpant': np.float64(4.362279871739047), 'rich': np.float64(7.193930820733286), 'comb': np.float64(4.251348311031766), 'cotton': np.float64(0.5814838578355617), 'give': np.float64(2.490477694837456), 'design': np.float64(1.6255933526563855), 'skin': np.float64(2.881437429162399), 'friendli': np.float64(4.449799249755603), 'fabric': np.float64(1.5888186252313647), 'itch': np.float64(6.115678473094656), 'free': np.float64(3.341075651483173), 'waistband': np.float64(3.9586786970689456), 'year': np.float64(3.9775116454020374), 'round': np.float64(1.07841171810501), 'use': np.float64(2.216642663193321), 'proudli': np.float64(4.446755107374376), 'india': np.float64(0.8549122464713206), 'elast': np.float64(2.9045

In [13]:
stemmer, stop_words = setup_preprocessing_tools()
df_indexed = df.set_index('pid')

def search_tfidf(query_text, inverted_index, idf_scores, df_docs, k=10):
    """
    Performs a ranked TF-IDF search for a given query.
    
    Args:
        query_text (str): The raw query string.
        inverted_index (dict): The inverted index.
        idf_scores (dict): The pre-calculated IDF scores for all terms.
        df_docs (pd.DataFrame): The DataFrame (indexed by 'pid') 
                                containing 'tfidf_vector' and 'doc_length'.
        k (int): The number of top results to return.

    Returns:
        list: A list of score, pid, title, brand, etc. tuples, sorted by score.
    """
    
    # Preprocess the query
    query_tokens = preprocess_query(query_text, stemmer, stop_words)
    
    # Find matching documents (Conjunctive/AND query)
    try:
        # Retrieve the set of pids for each query token
        doc_sets = [set(inverted_index[token]) for token in query_tokens if token in inverted_index]
        
        # If any token is not in the index, no docs will match an AND query
        if len(doc_sets) != len(query_tokens):
            print("One or more query terms not in index. No results.")
            return []

        # Find the intersection of all sets
        matching_pids = set.intersection(*doc_sets)
    
    except KeyError:
        # This handles if a token isn't in the index, though the check above is safer
        print("Query term not in index. No results.")
        return []

    if not matching_pids:
        print("No documents contain all query terms.")
        return []

    # Calculate the Query TF-IDF Vector and its length
    query_tf = calculate_log_tf(query_tokens)
    query_tfidf_vector = {}
    query_sum_of_squares = 0.0
    
    for term, tf in query_tf.items():
        if term in idf_scores:
            tfidf = tf * idf_scores[term]
            query_tfidf_vector[term] = tfidf
            query_sum_of_squares += tfidf**2
            
    query_length = np.sqrt(query_sum_of_squares)
    
    if query_length == 0:
        print("Query vector has no length (all terms unknown).")
        return []

    # Calculate Cosine Similarity for all matching documents
    scores = {}
    
    # Filter the main DataFrame to only the documents that matched
    # .loc is fast because we set the index to 'pid'
    matching_docs = df_docs.loc[list(matching_pids)]
    
    for pid, row in matching_docs.iterrows():
        doc_tfidf_vector = row['tfidf_vector']
        doc_length = row['doc_length']
        
        # Calculate Dot Product
        dot_product = 0.0
        # Iterate over the query vector, which is much smaller
        for term, query_tfidf_val in query_tfidf_vector.items():
            if term in doc_tfidf_vector:
                dot_product += query_tfidf_val * doc_tfidf_vector[term]
        
        # Calculate Cosine Similarity
        if doc_length > 0:
            scores[pid] = dot_product / (query_length * doc_length)
    
    # Sort and return the top K results
    sorted_results = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    
    final_results = []
    # Loop through the top k sorted (pid, score) tuples
    for pid, score in sorted_results[:k]:
        
        # Get the product data
        product_data = df_docs.loc[pid]
        
        # Append a dictionary with all the info we want
        final_results.append({
            'title': product_data['title'],
            'score': score,
            'pid': pid,
            'url': product_data['url'],
            'selling_price': product_data['selling_price'],
            'brand': product_data['brand_facet']
        })
        
    return final_results

## 1.2 BM25

In [14]:
# Compute document lengths and average length
df['L_d'] = df['tokens'].apply(len)
L_ave = df['doc_length'].mean()
N = len(df)
print(f"Number of documents: {N}, Average length: {L_ave:.2f}")

Number of documents: 28080, Average length: 18.88


In [15]:
df_counts = {term: len(postings) for term, postings in inverted_index.items()}

def bm25_idf(df_t, N):
    return np.log((N - df_t + 0.5) / (df_t + 0.5) + 1)

bm25_idf_scores = {t: bm25_idf(df_t, N) for t, df_t in df_counts.items()}


In [16]:
def search_bm25(query_text, inverted_index, bm25_idf, df_docs,
                k=10, k1=1.5, b=0.75):
    """
    BM25 ranking for a given query.

    Args:
        query_text (str): Raw query string.
        inverted_index (dict): Inverted index (term -> list of pids).
        bm25_idf (dict): Precomputed BM25 IDF scores.
        df_docs (pd.DataFrame): DataFrame indexed by 'pid', containing 'tokens' and 'doc_length'.
        k (int): Number of top results to return.
        k1, b (float): BM25 hyperparameters.

    Returns:
        List[dict]: Ranked list of top-k results with pid, title, brand, etc.
    """

    # 1) Preprocess query 
    query_tokens = preprocess_query(query_text, stemmer, stop_words)
    if not query_tokens:
        return []

    # 2) Candidate documents: union of postings of query terms 
    doc_sets = [set(inverted_index[t]) for t in query_tokens if t in inverted_index]
    if not doc_sets or len(doc_sets) < len(query_tokens):
        return []
    candidate_pids = set.intersection(*doc_sets)
    if not candidate_pids:
        return []

    # 3) Compute BM25 scores for candidate documents
    scores = {}
    L_ave = df_docs['doc_length'].mean()

    for pid in candidate_pids:
        row = df_docs.loc[pid]
        doc_tokens = row['tokens']
        Ld = row['doc_length']
        score = 0.0

        # term frequencies in this doc
        tf_doc = Counter(doc_tokens)

        for t in query_tokens:
            if t not in bm25_idf_scores:
                continue
            tf_td = tf_doc.get(t, 0)
            if tf_td == 0:
                continue

            idf = bm25_idf_scores[t]

            # BM25 term contribution
            denom = tf_td + k1 * ((1 - b) + b * (Ld / L_ave))
            term_score = idf * ((k1 + 1) * tf_td / denom)
            score += term_score

        if score > 0:
            scores[pid] = score

    if not scores:
        return []

    # --- Sort and format top-k results ---
    sorted_pids = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    results = []
    for pid, s in sorted_pids:
        row = df_docs.loc[pid]
        results.append({
            "title": row.get("title", ""),
            "score": float(s),
            "pid": pid,
            "brand": row.get("brand_facet", ""),
            "url": row.get("url", ""),
            "selling_price": row.get("selling_price", np.nan)
        })
    return results


## 1.3 Our Ranking Algorithm

In [17]:
# Pre computation of average rating per brand
# We use .fillna(2.5) to give a neutral rating to products without a rating
brand_avg_ratings = df.groupby('brand_facet')['average_rating'].mean().fillna(2.5).to_dict()

DEFAULT_BRAND_RATING = 2.5 

print(f"Computed average ratings for {len(brand_avg_ratings)} brands.")

Computed average ratings for 321 brands.


In [18]:
def search_custom(query_text, inverted_index, bm25_idf, df_docs, brand_ratings_map,
                        k=10, k1=1.5, b=0.75, 
                        w_rating=0.1, w_discount=0.05, w_price = 0.1, w_brand=0.05): #w_rating, w_discount, w_price, and w_brand are new hyperparameters. Can be tuned manually
    """
    Custom ranking: BM25 boosted by average_rating and discount. The idea is to use BM25 as a base score but adding the influence of
    average_rating, discount and price as multiplicative factors so that higher rated, more discounted products and cheaper products rank higher.

    Args:
        query_text (str): Raw query string.
        inverted_index (dict): Inverted index (term -> list of pids).
        bm25_idf (dict): Precomputed BM25 IDF scores.
        df_docs (pd.DataFrame): DataFrame indexed by 'pid', containing 'tokens', 'doc_length', 'average_rating', 'discount'.
        k (int): Number of top results to return.
        k1, b (float): BM25 hyperparameters.
        w_rating (float): Weight for average_rating boost.
        w_discount (float): Weight for discount boost.
        w_price (float): Weight for price decay.
        w_brand (float): Weight for brand average rating boost. (confidence in the brand even if the specific product has low rating or no rating)
    Returns:
        List[dict]: Ranked list of top-k results with pid, title, brand, etc
    """

    # Preprocessing
    query_tokens = preprocess_query(query_text, stemmer, stop_words)
    if not query_tokens:
        return []

    # Candidate documents
    doc_sets = [set(inverted_index[t]) for t in query_tokens if t in inverted_index]
    if not doc_sets or len(doc_sets) < len(query_tokens):
        return []
    candidate_pids = set.intersection(*doc_sets)
    if not candidate_pids:
        return []

    # Compute BM25 scores for candidate documents
    scores = {}
    L_ave = df_docs['doc_length'].mean()

    for pid in candidate_pids:
        row = df_docs.loc[pid]
        doc_tokens = row['tokens']
        Ld = row['doc_length']
        bm25_score = 0.0

        # term frequencies in this doc
        tf_doc = Counter(doc_tokens)

        for t in query_tokens:
            if t not in bm25_idf_scores:
                continue
            tf_td = tf_doc.get(t, 0)
            if tf_td == 0:
                continue

            idf = bm25_idf_scores[t]

            # BM25 term contribution
            denom = tf_td + k1 * ((1 - b) + b * (Ld / L_ave))
            term_score = idf * ((k1 + 1) * tf_td / denom)
            bm25_score += term_score
        
        # We obtain the numeric values of average_rating and discount (with 0 default if missing)
        rating = row.get('average_rating', 0)
        discount = row.get('discount', 0)
        price = row.get('selling_price', 0)
        brand = row.get('brand_facet', 'unknown')
        
        # Compute 'boosts'
        # We use np.log1p (log(1+x)), safe for values of 0

        # Boosts
        rating_boost = 1 + (w_rating * np.log1p(rating))

        discount_boost = 1 + (w_discount * np.log1p(discount))

        avg_brand_rating = brand_ratings_map.get(brand, DEFAULT_BRAND_RATING)
        brand_boost = 1 + (w_brand * np.log1p(avg_brand_rating))

        # Decay
        price_decay = 1 / (1 + (w_price * price)/1000)


        
        # Compute final score
        final_score = bm25_score * rating_boost * discount_boost * price_decay * brand_boost

        if final_score > 0:
            scores[pid] = final_score

    if not scores:
        return []

    # --- Sort and format top-k results ---
    sorted_pids = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    results = []
    for pid, s in sorted_pids:
        row = df_docs.loc[pid]
        results.append({
            "title": row.get("title", ""),
            "score": float(s),
            "pid": pid,
            "brand": row.get("brand_facet", ""),
            "url": row.get("url", ""),
            "selling_price": row.get("selling_price", np.nan),
            "average_rating": row.get("average_rating", np.nan),
            "discount": row.get("discount", np.nan) 
        })
    return results

## 1.4 Ranking Comparisons

In [19]:
#Test query list
five_query_list = ["blue shirt round neck machine wash", "solid men blue trackpants", "full sleeve casual shirt cotton", "women polo t-shirt", "white shirt cotton"]

In [20]:
for query in five_query_list:
    print(f"TF-IDF Results for query: '{query}'")
    results_tfidf = search_tfidf(query, inverted_index, idf_scores, df_indexed, k=5)
    for res in results_tfidf:
        print(res)
    print("\n")

    print(f"BM25 Results for query: '{query}'")
    results_bm25 = search_bm25(query, inverted_index, bm25_idf_scores, df_indexed, k=5)
    for res in results_bm25:
        print(res)
    print("\n")

    print(f"Custom Results for query: '{query}'")
    results_custom = search_custom(query, inverted_index, bm25_idf_scores, df_indexed, brand_avg_ratings , k=5, w_rating=0.4, w_discount=0.1, w_price=0.1, w_brand=0.1)
    for res in results_custom:
        print(res)
    print("\n\n---------------------\n\n")

TF-IDF Results for query: 'blue shirt round neck machine wash'
{'title': 'Printed Men Round Neck Blue T-Shirt', 'score': np.float64(0.7132183545459758), 'pid': 'TSHFXHAPHNFGX68Q', 'url': 'https://www.flipkart.com/rose-wear-printed-men-round-neck-blue-t-shirt/p/itm50b46d94567ee?pid=TSHFXHAPHNFGX68Q&lid=LSTTSHFXHAPHNFGX68QVMK2VG&marketplace=FLIPKART&srno=b_3_105&otracker=browse&fm=organic&iid=b959a0e1-6898-4d40-a5ed-197450a4fb57.TSHFXHAPHNFGX68Q.SEARCH&ssid=wrfonoocow0000001612412993603', 'selling_price': np.float64(215.0), 'brand': 'rose_we'}
{'title': 'Printed Men Round Neck Blue T-Shirt', 'score': np.float64(0.7047741417621183), 'pid': 'TSHFVSS3Q8GGG3XG', 'url': 'https://www.flipkart.com/xink-printed-men-round-neck-blue-t-shirt/p/itmda60b6ec09c7e?pid=TSHFVSS3Q8GGG3XG&lid=LSTTSHFVSS3Q8GGG3XGETIAW9&marketplace=FLIPKART&srno=b_1_3&otracker=browse&fm=organic&iid=376481ae-575a-402b-93e4-c1a1f55236d3.TSHFVSS3Q8GGG3XG.SEARCH&ssid=ql2g4kd1y80000001612415491360', 'selling_price': np.float64(39

# 2. Implementing  word2vec + cosine ranking score.

#  2.1 Loading and setting up


In [21]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [36]:
import gensim
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# 2.2 Function definition to generate a text with word2vec 

In [ ]:
sentences = df['processed_text'].apply(lambda x: word_tokenize(" ".join(x)))  
word2vec_model = gensim.models.Word2Vec(sentences, vector_size=300, window=5, min_count=1, workers=4)  # Train the model
word2vec_model.save("my_word2vec_model.model")

In [39]:
import numpy as np

def text_to_vector(text, word2vec_model):  #create single vector by averaging 
    
    words = word_tokenize(text)  # Tokenize the text
    vectors = []

    for word in words:
        try:
            vectors.append(word2vec_model.wv[word])  
        except KeyError:
            continue  # If the word is not in the model, we ignore it

    if vectors:
        return np.mean(vectors, axis=0)  # We return the average vector
    else:
        return np.zeros(word2vec_model.vector_size)  # zero vector if no valid words found


In [41]:
# Generate document vectors for each document
df['document_vector'] = df['processed_text'].apply(lambda x: text_to_vector(" ".join(x), word2vec_model))


In [42]:

# exampple to testt

query_vector = text_to_vector("blue shirt round neck machine wash", word2vec_model)

from sklearn.metrics.pairwise import cosine_similarity

document_vector = df.iloc[0]['document_vector']  # First document in this case 
cosine_sim = cosine_similarity([query_vector], [document_vector])

print(f"Cosine similarity between the query and the first document: {cosine_sim[0][0]}")


Cosine similarity between the query and the first document: 0.40017032623291016


# 2.3 Ranking 


In [52]:


def search_word2vec(query_text, inverted_index, df_docs, word2vec_model, k=20):
   

   
    query_tokens_search = preprocess_query(query_text, stemmer, stop_words) #the same cleaning we used in TF-IDF/BM25
    if not query_tokens_search:
        return []

    # sets of document IDs for each token in the query using the inverted index
    doc_sets = [set(inverted_index[t]) for t in query_tokens_search if t in inverted_index]
    if len(doc_sets) != len(query_tokens_search):
       
        return []


    # find the intersection of all document sets to get the candidate document IDs
    candidate_pids = set.intersection(*doc_sets)
    if not candidate_pids:
        return []

   
    # convert the raw query text into a vector using word2vec
    query_vector = text_to_vector(query_text.lower(), word2vec_model)

 
    scores = {} # to store the cosine similarity scores between the query and documents

    # for each candidate document, retrieve its document vector and compute cosine similarity
    for pid in candidate_pids:

        doc_vec = df_docs.loc[pid, 'document_vector']
        # doc_vec is already a numpy array from your previous step
        sim = cosine_similarity([query_vector], [doc_vec])[0][0]
        scores[pid] = sim

    if not scores:
        return []

   
    top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k] # sorting the results 

    results = []
    for pid, score in top:
        row = df_docs.loc[pid]
        results.append({
            "pid": pid,
            "score": float(score),
            "title": row.get("title", ""),
            "brand": row.get("brand_facet", ""),
            "url": row.get("url", ""),
            "selling_price": row.get("selling_price", np.nan)
        })

    return results


In [ ]:

# Set the DataFrame index by pid
df_indexed = df.set_index('pid')

# Define the queries again as in previous lab 
five_query_list = [
    "blue shirt round neck machine wash",
    "solid men blue trackpants",
    "full sleeve casual shirt cotton",
    "women polo t-shirt",
    "white shirt cotton"
]

# Iterate over each query in the list
for q in five_query_list:
    print("\n" + "="*80)
    print(f"Query: {q}")
    
    # Search using the Word2Vec model and inverted index
    results = search_word2vec(q, inverted_index, df_indexed, word2vec_model, k=20)
    
    # Check if results are empty
    if not results:
        print("No results found.")
        continue
    
    # Print the top 20 results
    for rank, r in enumerate(results, start=1):
        print(f"{rank:2d}. score={r['score']:.4f} | pid={r['pid']} | "
              f"title={r['title'][:80]} | brand={r['brand']} | price={r['selling_price']}")



Query: blue shirt round neck machine wash
 1. score=0.9169 | pid=TSHFVYHCYY5YPGZT | title=Solid Women Round Neck Dark Blue T-Shirt | brand=steenb | price=449.0
 2. score=0.9101 | pid=TSHFMF3NPPDCEPND | title=Printed Men Round Neck Blue T-Shirt | brand=arbo | price=426.0
 3. score=0.9101 | pid=TSHFNV35W6XMETKT | title=Printed Men Round Neck Blue T-Shirt | brand=tee_bud | price=399.0
 4. score=0.9101 | pid=TSHFME2EUDE7SNHV | title=Printed Men Round Neck Blue T-Shirt | brand=steenb | price=549.0
 5. score=0.9085 | pid=TSHFVXGQ73ZRG9AW | title=Printed Women Round Neck Blue T-Shirt | brand=mash_unlimit | price=329.0
 6. score=0.9080 | pid=TSHFZ3JEBFDR9XUE | title=Solid Men Round Neck Blue T-Shirt | brand=reeb | price=749.0
 7. score=0.9080 | pid=TSHFZ3JD8H4Z5NV7 | title=Solid Men Round Neck Blue T-Shirt | brand=reeb | price=749.0
 8. score=0.9047 | pid=TSHFZF6KDGVRSZKN | title=Printed Women Round Neck Blue T-Shirt | brand=adidas_origina | price=1264.0
 9. score=0.9042 | pid=TSHFZ3JDTYFSPMW